In [ ]:
import numpy as np
import h5py

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
import seaborn as sns

from src import io

In [ ]:
# set experiment date and number
exp_date = '250730'
exp_num = '1001'

experiment_path = io.get_experiment(exp_date, exp_num)
supervox_path = io.get_file(experiment_path, 'processed', '*supervoxels.h5')
labels_path = io.get_file(experiment_path, 'processed', '*labels.h5')
print(supervox_path, labels_path)

# open h5 and pull data
with h5py.File(labels_path, 'r') as labels_hf:
    cluster_labels = labels_hf['labels'][...]

with h5py.File(supervox_path,'r') as supervox_hf:
    ca_signal = supervox_hf['ca_signal'][...]
    fictrac_sp = supervox_hf['smoothed_speed'][...]
    fictrac_t = supervox_hf['fictrac_time'][...]
    camera_fr = supervox_hf['camera_fr'][...]
    scope_fr = supervox_hf['scope_fr'][...]
    brain_dim = supervox_hf['brain_dimensions'][...]

brain_size = brain_dim.reshape(-1)

# maui time conversions
frames = [x for x in range(1, ca_signal.shape[-1])]
volume_per_s = scope_fr/ca_signal.shape[0] # volume rate in Hz
maui_time = [0] + [x / volume_per_s for x in frames]

# fictrac time conversions
fic_time = fictrac_t / camera_fr

In [ ]:
# vis features
div_cmap = sns.diverging_palette(220, 20, as_cmap=True)
sing_cmap = sns.colo_palette('light:b', as_cmap=True)
color = 'b'

In [ ]:
# plotting z-scored df/F traces of individual ROIs
# plots highest correlated ROIs

fig0 = plt.figure(figsize=(20,20))
gs = GridSpec(nrows=3, ncols=1, figure=fig0)

ax0 = fig.addsubplot(gs[0:1,:])
for idx, cluster_idx in enumerate(sorted_pearson):
    trace = signal_reshape[cluster_idx, :] + (y_shift*idx)
    ax0.plot(maui_time, trace, color='c')
    mean_corr.append(pearson_arr[cluster_idx])
    if idx == roi_n:
        break
ax0.title(fr'average Pearson correlation coefficient = {np.mean(mean_corr):.2f}', loc = 'left')
ax0.xlim(xmin=0, xmax=max(maui_time))
ax0.ylabel('top 20 correlated rois')

ax1 = fig.addsubplot(gs[2, :])
ax1.plot(fic_time, fictrac_sp, color=color)
ax1.ylim(0,6)
ax1.xlabel('time(s)')
ax1.ylabel('instantaneous speed (rad/s)')

fig0.tight_layout()
plt.show

In [ ]:
# heatmap of dF/F for all ROIs
# need the array of sorted pearson correlations
n_toplot=50

# reshape ca_signal[slices, rois, df/f] to [rois, df/f]
all_traces = ca_signal.reshape(-1, ca_signal.shape[-1])

fig1 = plt.figure(figsize=(10,10))
sns.heatmap(all_traces[:n_toplot,:], cmap=cmap, yticklabels=False, xticklabels=False) # for a single roi [idx, np.newaxis]
fig1.title(fr'heatmap of df/F of {n_toplot} ROIs across time')
plt.show

# heat map sorted by correlation coefficient
sorted_traces = all_traces[sorted pearson]
fig2 = plt.figure(figsize=(10,10))
sns.heatmap(sorted_traces[:n_toplot,:], cmap=cmap, yticklabels=False, xticklabels=False) # for a single roi [idx, np.newaxis]
fig1.title(fr'heatmap of df/F of {n_toplot} ROIs across time - sorted by correlation')
plt.show

In [ ]:
# normalize color bar across all slices
norm_all = Normalize(vmin=np.min(corr_array), vmax=np.max(corr_array))

fig3 = plt.figure(figsize=(30,15))
gs = GridSpec(nrows=4, ncols=8, figure=fig3)
axes = gs.flatten() # idk if this will work, I just want a list of all the grid spaces

c_scale = []
for i, ax in enumerate(axes):
    c_scale.append(ax.imshow(corr_array[i].T, cmap=cmap, norm=norm_all))
        ax.set_title(f'Z-Slice {i}')
        ax.set_xticks([]) # Hide x-axis ticks for cleaner appearance
        ax.set_yticks([]) # Hide
fig3.colorbar(images[0], ax=ax, orientation='vertical', fraction=.1)
fig3.tight_layout()
plt.show()